# Video2LoRA: Self-Contained Zero-Visual-Token Inference Tutorial

This notebook provides a **completely self-contained, single-file tutorial** for running zero-visual-token inference with **Video2LoRA**. Under the hood, Video2LoRA utilizes a perceiver hypernetwork to read a video's intermediate activation features layer-by-layer and directly predict a set of Low-Rank Adaptation (LoRA) weights in a single forward pass.

Once the adapter weights are generated, they are injected into the base Vision-Language Model (VLM), which can then answer arbitrary text queries **without requiring the original visual tokens** in its context window.

All model architectures, helper functions, and dependencies are defined directly within this notebook so it can run independently of the main repository codebase.

### 🔗 Project Resources
- [**🌐 Project Website**](https://video2lora.github.io/)
- [**📄 arXiv Paper**](https://arxiv.org/abs/2606.04351)
- [**🤗 Hugging Face Checkpoints**](https://huggingface.co/MananSuri27/Video2LoRA-SmolVLM-ckpts)
- [**💻 GitHub Repository**](https://github.com/MananSuri27/video2lora)

## Step 0: Package Installation

First, we install all the required open-source packages in the environment.

In [ ]:
# Detect if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Setting up workspace and website repositories...")
    import os
    # 1. Clone the main Video2LoRA repository
    if not os.path.exists("/content/Video2LoRA"):
        os.system("git clone -b demo https://github.com/video2lora/Video2LoRA.git")
    # 2. Clone the website repository (which hosts all qualitative video assets)
    if not os.path.exists("/content/video2lora.github.io"):
        os.system("git clone https://github.com/video2lora/video2lora.github.io.git")
    # 3. Change directory to the codebase workspace
    os.chdir("/content/Video2LoRA")
    # 4. Create a symlink to the website's media directory so video paths resolve locally
    if not os.path.exists("media"):
        os.system("ln -s /content/video2lora.github.io/media media")
    print(f"Workspace directory changed to: {os.getcwd()}")

# Install necessary packages
!pip install torch torchvision torchaudio
!pip install transformers accelerate peft huggingface_hub decord av opencv-python matplotlib einops jaxtyping

## Step 1: Model Architectures & Utilities

Here, we define the full Video2LoRA model architecture classes (including the perceiver aggregator, hypernet heads, model load patches, and custom LoRA injection layers) directly inside the notebook. This makes the notebook completely independent of external python code files.

In [ ]:
import sys
import os
from pathlib import Path

# Add repository root and src directory to sys.path to enable local imports
repo_root = str(Path(os.getcwd()).resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
src_dir = str(Path(repo_root) / "src")
if os.path.exists(src_dir) and src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import torch
import cv2
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Optional

# Import custom Video2LoRA architectures and utilities from the repository modules
from ctx_to_lora.modeling.lora_layer import apply_lora_to_layers
from ctx_to_lora.modeling.lora_merger import combine_lora
from scripts.video2lora.train_smolvlm_stage1 import build_stage1_model
from scripts.video2lora.train_smolvlm_online import (
    prepare_smolvlm_inputs,
    extract_l2l_fused_text_features,
)

@dataclass
class TrainArgs:
    smolvlm_name_or_path: str
    train_manifest: str
    val_manifest: str
    output_dir: str
    lora_r: int = 16
    lora_dropout: float = 0.0
    target_modules: Optional[List[str]] = None
    latent_size: int = 512
    dropout_rate: float = 0.0
    n_latent_queries: int = 8
    num_blocks: int = 9
    num_self_attn_per_block: int = 0
    video_fps: Optional[float] = None
    max_frames: int = 12
    video_size_longest_edge: int = 384
    video_load_backend: str = "auto"
    internalization_prompt: str = "Internalize this video for later captioning."
    kl_weight: float = 0.0
    generation_max_new_tokens: int = 128

print("Video2LoRA imported architectures and utility functions successfully.")

## Step 2: Load and Inspect Manifest

Video2LoRA uses a standard JSONL manifest file for training and evaluation. Below, we define a sample manifest row matching the child watering plants example from the paper's qualitative evaluation.

In [ ]:
# Define the 6 qualitative examples from the Video2LoRA project website
examples = [
    {
        "id": "man-smoking-pipe",
        "video_path": "media/benchmarks/carebench/v_00014063_0.mp4",
        "dataset": "CaReBench: Caption",
        "prompt": "Describe the video in as much useful visual detail as possible. Include the main activity, visible people or objects, scene context, appearance, and any important visual details that help explain what is happening.",
        "target_text": "This video depicts a scene of a man lighting a pipe with a lighter. The man in the video is smoking a pipe held in his mouth, supported by his left hand, while his right hand grips the lighter. His right forearm features a large black tattoo. He has short, thick hair that is a deep brown color and is dressed in a loose-fitting black tank top. He is seated next to a window, which has a wooden frame and blue curtains, with a brick wall behind him and a wooden door on the right. The door has a square pattern, adorned with silver hinges and a doorknob. In the video, he first ignites the lighter with his right hand and then brings the flame to the pipe, holding it in that position for several seconds. Throughout this time, his left hand remains steady on the pipe, and his gaze is fixed intently on it, ensuring that the pipe is fully lit before setting the lighter down. He then continues to hold the pipe with his left hand and begins to smoke. The video is shot from the front, clearly illustrating how relaxed he is while smoking at home."
    },
    {
        "id": "child-watering-plants",
        "video_path": "media/benchmarks/carebench/v_00016555_0.mp4",
        "dataset": "CaReBench: Events",
        "prompt": "Describe the key visible events in chronological order. Include all important actions and changes you can observe, with enough detail to distinguish each event clearly.",
        "target_text": "Little boy watering plants outdoors; Using watering can to pour water into flower pot; Shifting camera angle from side view to rear view; Tapping edge of flower pot a few times; Setting down watering can"
    },
    {
        "id": "posture-alignment",
        "video_path": "media/benchmarks/plm/f522598789220c70_122_155.mp4",
        "dataset": "PLM-SGQA",
        "prompt": "Does this look like the same posture she's holding?",
        "target_text": "Yes, it appears you're mirroring the same posture. Your alignment, knee bend, and spine position match the demonstration, indicating proper form and engagement of the targeted muscle groups for optimal effectiveness and safety."
    },
    {
        "id": "dog-tugging",
        "video_path": "media/benchmarks/plm/b5bdb7f254cb1727_369_400.mp4",
        "dataset": "PLM-SGQA",
        "prompt": "Is he trying to tug?",
        "target_text": "Yes, your dog is likely inviting a tug-of-war game. Holding the toy in his mouth and possibly looking at you or wagging his tail indicates he's ready to engage in a playful tug."
    },
    {
        "id": "rainy-day",
        "video_path": "media/benchmarks/vidcapbench/132065802449.mp4",
        "dataset": "VidCapBench",
        "prompt": "What is the weather like in the scene? Answer only the question, in one sentence.",
        "target_text": "Rainy day."
    },
    {
        "id": "tarsier-creature",
        "video_path": "media/benchmarks/vidcapbench/Tarsier_20.mp4",
        "dataset": "VidCapBench",
        "prompt": "Which parts of the creature are highlighted in the video? Answer only the question, in one sentence.",
        "target_text": "A close-up of its face, eyes, and hair."
    }
]

In [ ]:
import os
import urllib.request

# Download the qualitative videos from the project website repo
print("Checking/downloading qualitative benchmark video files from project website...")
for item in examples:
    video_path = item["video_path"]
    os.makedirs(os.path.dirname(video_path), exist_ok=True)
    if not os.path.exists(video_path):
        url = f"https://video2lora.github.io/{video_path}"
        print(f"Downloading {video_path} from {url}...")
        try:
            urllib.request.urlretrieve(url, video_path)
            print("Download successful.")
        except Exception as e:
            print(f"Failed to download: {e}")
    else:
        print(f"Found local video file: {video_path}")

print(f"\nLoaded {len(examples)} qualitative examples successfully.")

## Step 3: Visualize Video Frames

Let's write a utility function to extract and display keyframes from the input video file using OpenCV and Matplotlib.

In [ ]:
def show_video_frames(video_path, num_frames=4):
    """
    Helper function to load keyframes from a video path and plot them in a grid.
    """
    if not os.path.exists(video_path):
        print(f"Video file not found at: {video_path} (Please provide a valid video to inspect).")
        return
        
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    
    fig, axes = plt.subplots(1, num_frames, figsize=(16, 4))
    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[i].imshow(frame)
            axes[i].axis('off')
            axes[i].set_title(f"Frame {idx}")
        else:
            axes[i].text(0.5, 0.5, "Frame Read Error", ha="center", va="center")
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()
    cap.release()

# Visualize the first qualitative example (man smoking a pipe)
print(f"Visualizing keyframes for Example 1: {examples[0]['id']}")
show_video_frames(examples[0]["video_path"], num_frames=4)

## Step 4: Download Video2LoRA Checkpoint

We use the `huggingface_hub` client to download the 2.2B Video2LoRA checkpoint weights (`video2lora-smolvlm2-2.2b-best-ce.pt`) directly from Hugging Face.

In [ ]:
print("Downloading 2.2B Video2LoRA Checkpoint from Hugging Face...")

checkpoint_dir = Path("checkpoints/Video2LoRA-SmolVLM-ckpts")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Download checkpoint weights programmatically
ckpt_path = hf_hub_download(
    repo_id="MananSuri27/Video2LoRA-SmolVLM-ckpts",
    filename="video2lora-smolvlm2-2.2b-best-ce.pt",
    local_dir=str(checkpoint_dir)
)

print(f"Checkpoint successfully downloaded to: {ckpt_path}")

## Step 5: Load Base VLM & Initialize Modulated Model

Configure `TrainArgs` matching the 2.2B preset, load the base SmolVLM2 model, processor, and tokenizer, and load the hypernetwork checkpoint.

In [ ]:
# Set configuration arguments for the 2.2B SmolVLM2 model preset
train_args = TrainArgs(
    smolvlm_name_or_path="HuggingFaceTB/SmolVLM2-2.2B-Instruct",
    train_manifest="",
    val_manifest="",
    output_dir="",
    lora_r=16,
    lora_dropout=0.0,
    target_modules=["down_proj"],
    latent_size=512,
    dropout_rate=0.0,
    n_latent_queries=8,
    num_blocks=9,
    num_self_attn_per_block=0,
    video_fps=None,
    max_frames=12,
    video_size_longest_edge=384,
    video_load_backend="auto",
    internalization_prompt="Internalize this video for later captioning.",
    kl_weight=0.0,
    generation_max_new_tokens=128
)

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize SmolVLM2 base model, processor, and tokenizer
print("Loading SmolVLM2 model (2.2B-Instruct)...")
model, raw_model, processor, tokenizer = build_stage1_model(train_args, device=device)

# 2. Load the hypernetwork weights
print("Loading hypernetwork weights...")
state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict)

# Place models in evaluation mode
model.to(device)
model.eval()
raw_model.eval()

print("Model load complete.")

## Step 6: Parametric Video Internalization

We process the video frames through the VLM visual encoder to extract activations, then generate the custom LoRA adapter weights in a single pass of the hypernetwork.

In [ ]:
def run_internalization(example, model, raw_model, processor, train_args, device):
    """
    Extracts visual features layer-by-layer and generates the dynamic LoRA adapter.
    """
    internalize_messages = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "video", "path": example["video_path"]},
                    {"type": "text", "text": train_args.internalization_prompt}
                ]
            }
        ]
    ]

    vlm_inputs = prepare_smolvlm_inputs(
        processor,
        internalize_messages,
        device,
        video_fps=train_args.video_fps,
        max_frames=train_args.max_frames,
        video_size_longest_edge=train_args.video_size_longest_edge,
        video_load_backend=train_args.video_load_backend
    )

    ctx_features, ctx_attn_mask, ctx_position_ids = extract_l2l_fused_text_features(
        raw_model,
        vlm_inputs,
        num_target_layers=model.hypernet.n_layers
    )

    generated_loras, _ = model.generate_weights(
        ctx_ids=None,
        ctx_features=ctx_features,
        ctx_attn_mask=ctx_attn_mask,
        ctx_position_ids=ctx_position_ids
    )

    generated_loras = combine_lora(
        generated_loras,
        torch.ones(1, dtype=torch.int32, device=device),
        lora_bias=model.hypernet.get_head_bias() if model.hypernet.config.use_bias else None
    )

    return generated_loras

# Example run of internalization for child watering plants (index 1)
print(f"Running internalization for Example 2: {examples[1]['id']}...")
sample_loras = run_internalization(examples[1], model, raw_model, processor, train_args, device)
print("Internalization successful.")

## Step 7: Run QA Inference (Base Model vs Video2LoRA)

To perform a fair comparison, we first run inference with the **Base Model** (which requires processing the full visual video tokens in context). Afterwards, we run inference with **Video2LoRA** (which bypasses visual tokens completely and queries the model using only text, utilizing the dynamic LoRA adapter).

In [ ]:
def run_inference(example, generated_loras, model, raw_model, processor, tokenizer, train_args, device):
    """
    Runs inference comparing base model (using visual tokens) vs Video2LoRA (zero visual tokens).
    """
    # === Part A: Base Model Inference (with visual tokens) ===
    base_messages = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "video", "path": example["video_path"]},
                    {"type": "text", "text": example["prompt"]}
                ]
            }
        ]
    ]

    base_inputs = prepare_smolvlm_inputs(
        processor,
        base_messages,
        device,
        video_fps=train_args.video_fps,
        max_frames=train_args.max_frames,
        video_size_longest_edge=train_args.video_size_longest_edge,
        video_load_backend=train_args.video_load_backend
    )

    base_generated_ids = raw_model.generate(
        **base_inputs,
        max_new_tokens=train_args.generation_max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    base_prediction = tokenizer.decode(
        base_generated_ids[0][base_inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # === Part B: Video2LoRA Inference (zero visual tokens) ===
    apply_lora_to_layers(
        model.base_model,
        model.hypernet.layer_indices,
        generated_loras,
        torch.ones(1, dtype=torch.int32, device=device),
        position_ids=None
    )

    prompt_ids = tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": [{"type": "text", "text": example["prompt"]}]
            }
        ],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)

    generated_ids = model.base_model.generate(
        input_ids=prompt_ids,
        attention_mask=torch.ones_like(prompt_ids),
        max_new_tokens=train_args.generation_max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    video2lora_prediction = tokenizer.decode(
        generated_ids[0][prompt_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Reset model LoRA patches
    model.reset()

    return base_prediction, video2lora_prediction

# Run comparison inference on the sample example
print(f"Running inference for Example 2: {examples[1]['id']}...")
base_prediction, video2lora_prediction = run_inference(
    examples[1], sample_loras, model, raw_model, processor, tokenizer, train_args, device
)
print(f"\nPredictions Generated:\nBase Model: {base_prediction}\nVideo2LoRA: {video2lora_prediction}")

## Step 8: Qualitative Comparison Dashboard

Display the comparison between the Base Model and Video2LoRA using the dynamically generated prediction outputs in a custom IPython HTML layout matching the evaluation layout.

In [ ]:
from IPython.display import HTML, display

def display_comparison(
    video_path,
    question_prompt,
    ground_truth,
    base_model_output,
    video2lora_output,
    dataset_name
):
    """
    Renders a beautifully styled comparison board with local HTML5 video player.
    """
    html_content = f"""
    <div style="font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; max-width: 900px; margin: 20px auto; color: #1e293b; background-color: #f8fafc; padding: 24px; border-radius: 16px; box-shadow: 0 4px 20px rgba(0,0,0,0.05);">
        
        <!-- Header Source Badge -->
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px;">
            <h3 style="margin: 0; font-size: 18px; font-weight: 700; color: #0f172a;">Qualitative Comparison</h3>
            <span style="background-color: #e2e8f0; color: #475569; padding: 4px 10px; border-radius: 9999px; font-size: 12px; font-weight: 600;">{dataset_name}</span>
        </div>

        <!-- Local HTML5 Video Player -->
        <div style="text-align: center; margin-bottom: 20px; background: #000; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
            <video src="{video_path}" controls loop muted autoplay style="width: 100%; max-height: 400px; object-fit: contain; display: block; margin: 0 auto;"></video>
        </div>
        
        <!-- Question Prompt Card -->
        <div style="background-color: #f0f4ff; border: 1px solid #dbeafe; border-radius: 8px; padding: 16px; margin-bottom: 16px;">
            <h5 style="margin: 0 0 6px 0; color: #1e40af; text-transform: uppercase; font-size: 11px; letter-spacing: 0.05em; font-weight: 700;">QUESTION PROMPT</h5>
            <p style="margin: 0; font-size: 15px; line-height: 1.5; color: #1e293b;">{question_prompt}</p>
        </div>
        
        <!-- Ground Truth Card -->
        <div style="background-color: #fefbeb; border: 1px solid #fef3c7; border-radius: 8px; padding: 16px; margin-bottom: 16px;">
            <h5 style="margin: 0 0 6px 0; color: #854d0e; text-transform: uppercase; font-size: 11px; letter-spacing: 0.05em; font-weight: 700;">GROUND TRUTH</h5>
            <p style="margin: 0; font-size: 15px; line-height: 1.5; color: #1e293b; font-weight: 500;">{ground_truth}</p>
        </div>
        
        <!-- Model Comparisons Grid -->
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
            
            <!-- Base Model Card -->
            <div style="background-color: #fff; border: 1.5px solid #ea580c; border-radius: 8px; padding: 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.02);">
                <h5 style="margin: 0 0 6px 0; color: #ea580c; text-transform: uppercase; font-size: 11px; letter-spacing: 0.05em; font-weight: 700;">BASE MODEL (with visual tokens)</h5>
                <p style="margin: 0; font-size: 14px; line-height: 1.5; color: #334155;">{base_model_output}</p>
            </div>
            
            <!-- Video2LoRA Card -->
            <div style="background-color: #f0fdf4; border: 1.5px solid #16a34a; border-radius: 8px; padding: 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.02);">
                <h5 style="margin: 0 0 6px 0; color: #16a34a; text-transform: uppercase; font-size: 11px; letter-spacing: 0.05em; font-weight: 700;">VIDEO2LORA (zero visual tokens)</h5>
                <p style="margin: 0; font-size: 14px; line-height: 1.5; color: #1e293b; font-weight: 500;">{video2lora_output}</p>
            </div>
            
        </div>
        
    </div>
    """
    display(HTML(html_content))

# Loop through all 6 qualitative examples, run internalization + inference, and display the dashboards
print("Processing and running qualitative comparison inference for all 6 examples...")
for idx, example in enumerate(examples):
    print(f"\n--- Running Example {idx+1}/{len(examples)}: {example['id']} ---")
    # 1. Internalize
    loras = run_internalization(example, model, raw_model, processor, train_args, device)
    # 2. Run inference
    base_pred, v2l_pred = run_inference(example, loras, model, raw_model, processor, tokenizer, train_args, device)
    # 3. Render dashboard
    display_comparison(
        video_path=example["video_path"],
        question_prompt=example["prompt"],
        ground_truth=example["target_text"],
        base_model_output=base_pred,
        video2lora_output=v2l_pred,
        dataset_name=example["dataset"]
    )